In [ ]:
from pathlib import Path

import polars as pl

In [ ]:
dir = Path(r".../geotab-sf-tofromwithin")
glob_pattern = "OriginDestinationMatrix*.csv"
# number of columns are not fully the same, so
cols = ["DateFrom1", "DateTo1", "DaysOfWeek", "TimeFrom", "TimeTo"]
time_period_df = pl.read_csv(dir / "time_period_expansion_factor-Bayview.csv")

In [ ]:
dfs = (
    pl.read_csv(f).with_columns(
        # 2 cols have numbers with ',' so need to cast them explicitly to float
        pl.col("DailyJourneyCountAvg", "DurationAtDestinationStdev")
        # HOTFIX if all values in col < 1000, it would already be a float,
        # cast as str first to allow consistent handling
        .cast(pl.Utf8)
        .str.replace_all(",", "")
        .cast(pl.Float64)
    )
    for f in Path(dir).glob(glob_pattern)
)
df = pl.concat(dfs, how="diagonal", rechunk=True).join(
    time_period_df, on=["TimeFrom", "TimeTo"]
)

In [ ]:
df_by_countyline_long = (
    df.select(
        "DateFrom1",
        "DateTo1",
        "DaysOfWeek",
        "time_period",
        "TimeFrom",
        "TimeTo",
        "TimeComponent",
        "OriginZoneId",
        "OriginZoneDescription",
        "DestinationZoneId",
        "DestinationZoneDescription",
        "DailyJourneyCountAvg",
        "expansion_factor",
        "hours_in_time_period",
    )
    .with_columns(
        # Geotab returns exSF-exSF entries even though we didn't buy the data
        # and the values are probably wrong
        DailyJourneyCountAvg=pl.when(
            (pl.col("OriginZoneDescription") != "San Francisco")
            & (pl.col("DestinationZoneDescription") != "San Francisco")
        )
        .then(None)
        .otherwise(pl.col("DailyJourneyCountAvg"))
    )
    .with_columns(
        daily_journeys_expanded=(
            pl.col("DailyJourneyCountAvg") * pl.col("expansion_factor")
        )
    )
    .with_columns(
        hourly_journeys_expanded=(
            pl.col("daily_journeys_expanded") / pl.col("hours_in_time_period")
        )
    )
    .drop("hours_in_time_period")
)

In [ ]:
group_by = [
    "DateFrom1",
    "DateTo1",
    "DaysOfWeek",
    "time_period",
    "TimeFrom",
    "TimeTo",
    "TimeComponent",
    "OriginZoneDescription",
    "DestinationZoneDescription",
]
sort = ["time_period", "OriginZoneDescription", "DestinationZoneDescription"]
df_long = (
    df_by_countyline_long.with_columns(
        pl.when(pl.col(c) != "San Francisco")
        .then(pl.lit("ex SF"))
        .otherwise(pl.lit("San Francisco"))
        .alias(c)
        for c in ["OriginZoneDescription", "DestinationZoneDescription"]
    )
    .drop("OriginZoneId", "DestinationZoneId")
    .group_by(group_by)
    .agg(
        # to have empty/null sums return None instead of 0 (from pl.sum docs)
        pl.when(pl.col("daily_journeys_expanded").count() > 0).then(
            pl.sum("daily_journeys_expanded")
        ),
        pl.when(pl.col("hourly_journeys_expanded").count() > 0).then(
            pl.sum("hourly_journeys_expanded")
        ),
    )
    .sort(sort)
)

In [ ]:
df_by_countyline_long.write_csv(
    dir / "sf-tofromwithin-gvwr6to8-tuetothu-long-by_countyline.csv"
)
df_long.write_csv(dir / "sf-tofromwithin-gvwr6to8-tuetothu-long.csv")